In [1]:
import numpy as np
import pandas as pd
from itertools import product
from SALib.sample import latin, sobol
import os

print("Current working directory:", os.getcwd())


Current working directory: C:\Workspace\Post_Doc_Works_NTNU\Projects\2_SWE_Velocity_LV_Filling_Pressure_Digital_Twin\3_Codes


In [3]:
# Parameter definitions: [min, max, increment]
param_definitions = {
    'K_ES':   [1, 20, 0.5],
    'K_ED':   [1.0, 3, 0.5],
    't_peak': [0.2, 0.5, 0.05],
    'R_mv':  [0.01, 0.09, 0.01],
    'R_sys': [0.5, 2.5, 0.25],
    'C_ao':  [0.2, 1, 0.05],
    'C_sv':  [3.0, 30, 1],
    'Z_ao':  [0.01, 0.1, 0.01],
    'V_tot': [100, 500, 25]
}

param_names = list(param_definitions.keys())
bounds = np.array([[low, high] for low, high, _ in param_definitions.values()])


In [4]:
print("Generating full grid (small scale test)...")
grid_values = [np.arange(start, stop + step / 2, step) for start, stop, step in param_definitions.values()]
# Warning: reduce full grid for testing!
reduced_grid_values = [g[:2] for g in grid_values]  # take only first 2 values of each
full_grid = np.array(list(product(*reduced_grid_values)))
df_full = pd.DataFrame(full_grid, columns=param_names)
df_full.to_csv("full_grid_samples.csv", index=False)
print(f"Full grid saved: {df_full.shape[0]} rows")


Generating full grid (small scale test)...
Full grid saved: 512 rows


In [5]:
N = 2000
print("Generating random samples...")
random_samples = np.random.rand(N, len(param_names))
for i in range(len(param_names)):
    low, high = bounds[i]
    random_samples[:, i] = low + random_samples[:, i] * (high - low)
df_rand = pd.DataFrame(random_samples, columns=param_names)
df_rand.to_csv("random_samples.csv", index=False)

print("Generating LHS samples...")
lhs_samples = latin.sample(problem={'num_vars': 9, 'names': param_names, 'bounds': bounds.tolist()}, N=N)
df_lhs = pd.DataFrame(lhs_samples, columns=param_names)
df_lhs.to_csv("lhs_samples.csv", index=False)

print("Generating Sobol samples...")
sobol_samples = sobol.sample(problem={'num_vars': 9, 'names': param_names, 'bounds': bounds.tolist()}, N=2048)
df_sobol = pd.DataFrame(sobol_samples, columns=param_names)
df_sobol.to_csv("sobol_samples.csv", index=False)

print("✅ All CSV files saved.")


Generating random samples...
Generating LHS samples...
Generating Sobol samples...
✅ All CSV files saved.
